In [ ]:
from supabase import create_client
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, confusion_matrix

SUPABASE_URL     = os.environ["SUPABASE_URL"]
SUPABASE_KEY     = os.environ["SUPABASE_KEY"]
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)


def select_supabase(table, columns="*", filters=None, batch_size=1000):
     all_rows = []
     start = 0

     while True:
         query = supabase.table(table).select(columns).range(start, start + batch_size - 1)

         if filters:
             for f in filters:
                 query = query.filter(*f)

         resp = query.execute()
         rows = resp.data or []

         if not rows:
             break

         all_rows.extend(rows)

         if len(rows) < batch_size:
             break

         start += batch_size

     return all_rows

In [ ]:
response = select_supabase("servo_prices", columns="updated_at, price", filters=[("fuel_type", "eq", "U91"), ('station_id','eq','QyeyP5dO1ET4B+K47GlG67+tWN7coqUEiJPpf6NtDY8=')], batch_size=1000)

dfp = pd.DataFrame(response)

dfp.loc[pd.to_datetime(dfp["updated_at"], utc=True).dt.tz_convert("Australia/Sydney").between(datetime(2026, 5, 1, tzinfo=timezone.utc), datetime(2026, 7, 2, tzinfo=timezone.utc)), 'price'] += 16

dfp["updated_at_melb"] = pd.to_datetime(dfp["updated_at"], utc=True).dt.tz_convert("Australia/Sydney").dt.date.astype('str')
dfp["updated_at_melb_wd"] = pd.to_datetime(dfp["updated_at"], utc=True).dt.tz_convert("Australia/Sydney").dt.weekday

In [ ]:
dfagg = pd.DataFrame(dfp.groupby(['updated_at_melb','updated_at_melb_wd'])['price'].median()).reset_index()

In [ ]:

response = select_supabase("market_data", columns="date,metric, value", filters=[("metric", "in", "(brent_crude,usd_aud)")], batch_size=1000)

dfm = pd.DataFrame(response)
dfm = dfm.pivot(index="date", columns="metric", values="value").reset_index()
dfm.ffill(inplace=True) # Fill in missing values with the last known value
dfm.head()

In [ ]:
df = dfm.merge(dfagg, left_on='date', right_on="updated_at_melb", how="left")

In [ ]:
# --- Brent crude features ---
for lag in range(1, 8):
    df[f"brent_lag{lag}"] = df["brent_crude"].shift(lag)
    
df["brent_ma3"]   = df["brent_crude"].shift(1).rolling(3).mean()
df["brent_ma7"]   = df["brent_crude"].shift(1).rolling(7).mean()
# Trend: is brent rising? (lag1 minus lag5 — the 5-day lag you observed)
df["brent_trend"] = df["brent_crude"].shift(1) - df["brent_crude"].shift(5)

# --- U91 price features ---
df["u91_lag1"]     = df["price"].shift(1)
df["u91_lag2"]     = df["price"].shift(2)
# Momentum: is the local price already moving?
df["u91_momentum"] = df["price"].shift(1) - df["price"].shift(3)


#df["days_since_cycle_low"] = days_since_cycle_low(df['price'],  3)
df["is_weekend"]   = (df["updated_at_melb_wd"] >= 5).astype(int)


# One-hot encode day of week
for d in range(7):
    df[f"dow_{d}"] = (df["updated_at_melb_wd"] == d).astype(int)
    
df.set_index("date", inplace=True)
drop_cols = ["updated_at_melb", "updated_at_melb_wd"]

df.drop(columns=drop_cols, inplace=True)

df.tail(5)

In [ ]:
df["target"] = (df["price"].shift(-3) > df["price"]).astype(int)
df = df.dropna(subset=["price", "target"])

feature_cols = [c for c in df.columns if c not in ["price", "target"]]
df = df.dropna(subset=feature_cols)
 
features = df[feature_cols]
target   = df["target"]
print(f"Features shape: {features.shape}, Target shape: {target.shape}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42, stratify=target
)


#model = XGBClassifier(**model_params)
model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
#pd.Series(model.feature_importances_, index=features.columns).sort_values(ascending=False).plot(kind="barh", figsize=(10, 6))

In [ ]:
model.predict(pd.DataFrame(df[features.columns].iloc[-1]).T)